# Udacity Project 2: Operationalizing an ML Pipeline
## Azure Machine Learning Python SDK v2 Master Execution Notebook

**Master execution rule:** `nb_setup.ipynb` is the only controller for the Project 2 Azure Machine Learning workflow. Modules under `src/`, `deployment/`, `model_inspection/`, and `pipeline/` expose reusable functions. Do not run project workflow scripts from Terminal.

### Confirmed resources
- Resource group: `rg-udacity-azureml`
- Workspace: `udacity-project`
- Compute Instance: `ci-udacity-project`
- Compute Cluster: `cpu-cluster`
- Raw data: `bankmarketing:1` (`uri_file`)
- Training data: `bankmarketing-mltable:1` (`mltable`)
- AutoML experiment: `bankmarketing-automl-sdk-v2`
- Target column: `y`

### Operating rules
1. Search and validate before creating any resource.
2. Reuse valid completed resources.
3. Creation requires an explicit Boolean switch.
4. Never silently replace successful immutable assets.
5. Keep all cleanup disabled until screenshots and screencast are secured.
6. Do not print subscription IDs, tenant IDs, keys, tokens, credentials, or full Azure resource IDs.


In [1]:
# %pip install --upgrade "azureml-mlflow>=1.61.0"

In [2]:
# %pip install --upgrade --force-reinstall --no-deps "mlflow-skinny==3.18.0"

In [3]:
from importlib.metadata import version

print("azureml-mlflow:", version("azureml-mlflow"))
print("mlflow:", version("mlflow"))

azureml-mlflow: 1.62.0.post6
mlflow: 3.18.0


# PART 0 — FOUNDATION

## 0. Configuration and safety switches

**Objective:** Define confirmed names, cost controls, and explicit resource-creation gates.

In [4]:
from pathlib import Path

RESOURCE_GROUP = "rg-udacity-azureml"
WORKSPACE_NAME = "udacity-project"
COMPUTE_INSTANCE_NAME = "ci-udacity-project"
COMPUTE_NAME = "cpu-cluster"
RAW_DATA_NAME = "bankmarketing"
RAW_DATA_VERSION = "1"
MLTABLE_NAME = "bankmarketing-mltable"
MLTABLE_VERSION = "1"
TARGET_COLUMN = "y"

ALLOW_LOCAL_MLTABLE_REFRESH = False
ALLOW_MLTABLE_REGISTRATION = False
ALLOW_NEW_AUTOML_JOB = False
ALLOW_MODEL_REGISTRATION = False
ALLOW_ENDPOINT_CREATION = False
ALLOW_DEPLOYMENT_CREATION = False
ALLOW_TRAFFIC_UPDATE = False
ALLOW_MONITORING_REQUESTS = True
ALLOW_BENCHMARK = False
ALLOW_NEW_PIPELINE_JOB = False
ALLOW_COMPONENT_REGISTRATION = True
ALLOW_BATCH_ENDPOINT_CREATION = True
ALLOW_BATCH_DEPLOYMENT_CREATION = True
ALLOW_DEFAULT_DEPLOYMENT_UPDATE = True
ALLOW_BATCH_INVOCATION = False
ALLOW_SCHEDULE_CREATION = False
ALLOW_SCHEDULE_ENABLE = False
ALLOW_CLEANUP = False

EXPECTED_MIN_NODES = 0
EXPECTED_MAX_NODES = 1
EXPECTED_IDLE_SECONDS = 120

print("Safety switches loaded. Review before continuing.")

Safety switches loaded. Review before continuing.


## 1. Repository-root validation

**Objective:** Locate the confirmed repository without a broad recursive search.

In [5]:
import os

CONFIRMED_REPO_ROOT = Path(
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/"
    "ci-udacity-project/code/Users/ml.slayer.2019/"
    "azure-ml-bankmarketing-sdk-v2"
)
candidates = [CONFIRMED_REPO_ROOT, Path.cwd(), Path.cwd().parent if Path.cwd().name == "notebooks" else None]
repo_root = next((p.resolve() for p in candidates if p and p.exists() and (p / "Project_2_Master_Notebook_SDK_v2_Working.ipynb").exists() and (p / "src").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Validated repository root not found. Synchronize the confirmed Git branch, then rerun.")
os.chdir(repo_root)
print("Repository root:", repo_root)
print("Current working directory:", Path.cwd())

Repository root: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2
Current working directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2


## 2. Folder and required-file validation

**Objective:** Stop clearly when repository artifacts are missing; do not create source folders silently.

In [6]:
# ============================================================
# COMPLETE REPOSITORY FILE VALIDATION
# ============================================================
required_paths = [
	# Repository
	repo_root / "Project_2_Master_Notebook_SDK_v2_Working.ipynb",
	repo_root / "README.md",
	repo_root / "requirements.txt",
	repo_root / ".gitignore",
	# Data
	repo_root / "data" / "bankmarketing_mltable",
	repo_root / "data" / "bankmarketing_mltable" / "MLTable",
	# Source modules
	repo_root / "src",
	repo_root / "src" / "__init__.py",
	repo_root / "src" / "settings.py",
	repo_root / "src" / "connect_workspace.py",
	repo_root / "src" / "prepare_data.py",
	repo_root / "src" / "run_automl.py",
	repo_root / "src" / "register_best_model.py",
	repo_root / "src" / "deploy_online.py",
	# Deployment modules
	repo_root / "deployment",
	repo_root / "deployment" / "__init__.py",
	repo_root / "deployment" / "request_builder.py",
	repo_root / "deployment" / "logging.py",
	repo_root / "deployment" / "scoring.py",
	repo_root / "deployment" / "monitoring.py",
	# Model inspection
	repo_root / "model_inspection",
	repo_root / "model_inspection" / "__init__.py",
	repo_root / "model_inspection" / "inspection.py",
	# Pipeline modules
	repo_root / "pipeline",
	repo_root / "pipeline" / "__init__.py",
	repo_root / "pipeline" / "pipeline_definition.py",
	repo_root / "pipeline" / "component_registration.py",
	repo_root / "pipeline" / "batch_deployment.py",
	repo_root / "pipeline" / "schedule_management.py",
	# Pipeline command steps
	repo_root / "pipeline" / "steps",
	repo_root / "pipeline" / "steps" / "validate_data.py",
	repo_root / "pipeline" / "steps" / "train_model.py",
]
missing_paths = [
	path
	for path in required_paths
	if not path.exists()
]
if missing_paths:
	missing_text = "\n".join(
    	f"- {path.relative_to(repo_root)}"
    	for path in missing_paths
	)
	raise FileNotFoundError(
    	"Required repository files are missing:\n"
    	+ missing_text
	)
print("Complete repository validation: PASSED")
for path in required_paths:
	print("OK:", path.relative_to(repo_root))


Complete repository validation: PASSED
OK: Project_2_Master_Notebook_SDK_v2_Working.ipynb
OK: README.md
OK: requirements.txt
OK: .gitignore
OK: data/bankmarketing_mltable
OK: data/bankmarketing_mltable/MLTable
OK: src
OK: src/__init__.py
OK: src/settings.py
OK: src/connect_workspace.py
OK: src/prepare_data.py
OK: src/run_automl.py
OK: src/register_best_model.py
OK: src/deploy_online.py
OK: deployment
OK: deployment/__init__.py
OK: deployment/request_builder.py
OK: deployment/logging.py
OK: deployment/scoring.py
OK: deployment/monitoring.py
OK: model_inspection
OK: model_inspection/__init__.py
OK: model_inspection/inspection.py
OK: pipeline
OK: pipeline/__init__.py
OK: pipeline/pipeline_definition.py
OK: pipeline/component_registration.py
OK: pipeline/batch_deployment.py
OK: pipeline/schedule_management.py
OK: pipeline/steps
OK: pipeline/steps/validate_data.py
OK: pipeline/steps/train_model.py


## 3. Python import-path setup

**Objective:** Make repository packages importable after a kernel restart.

In [7]:
import sys, importlib
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
importlib.invalidate_caches()
print("Repository on sys.path:", str(repo_root) in sys.path)

Repository on sys.path: True


In [8]:

# ============================================================
# PROJECT MODULE IMPORT VALIDATION
# ============================================================

import importlib
import importlib.util
import sys
from pathlib import Path

# Ensure the repository has import precedence.
repo_root_text = str(repo_root.resolve())

sys.path = [
    path
    for path in sys.path
    if str(Path(path or ".").resolve()) != repo_root_text
]

sys.path.insert(0, repo_root_text)

# Clear local package caches after creating or changing files.
local_packages = (
    "src",
    "deployment",
    "model_inspection",
    "pipeline",
)

cached_local_modules = [
    module_name
    for module_name in list(sys.modules)
    if module_name in local_packages
    or module_name.startswith(
        tuple(f"{package}." for package in local_packages)
    )
]

for module_name in cached_local_modules:
    del sys.modules[module_name]

importlib.invalidate_caches()

# Required project modules.
required_modules = [
    # Source
    "src.settings",
    "src.connect_workspace",
    "src.prepare_data",
    "src.run_automl",
    "src.register_best_model",
    "src.deploy_online",

    # Deployment
    "deployment.request_builder",
    "deployment.logging",
    "deployment.scoring",
    "deployment.monitoring",

    # Model inspection
    "model_inspection.inspection",

    # Pipeline
    "pipeline.pipeline_definition",
    "pipeline.component_registration",
    "pipeline.batch_deployment",
    "pipeline.schedule_management",
]

missing_modules = []

for module_name in required_modules:
    specification = importlib.util.find_spec(module_name)

    if specification is None:
        missing_modules.append(module_name)
        print("MISSING:", module_name)
    else:
        print(
            "FOUND:",
            module_name,
            "->",
            specification.origin,
        )

if missing_modules:
    raise ModuleNotFoundError(
        "Required project modules are not discoverable:\n"
        + "\n".join(
            f"- {module_name}"
            for module_name in missing_modules
        )
    )

print("\nProject module discovery: PASSED")

FOUND: src.settings -> /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/src/settings.py
FOUND: src.connect_workspace -> /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/src/connect_workspace.py
FOUND: src.prepare_data -> /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/src/prepare_data.py
FOUND: src.run_automl -> /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/src/run_automl.py
FOUND: src.register_best_model -> /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/src/register_best_model.py
FOUND: src.deploy_online -> /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmark

## 4. Package and environment validation

**Objective:** Confirm required SDK v2 packages in the selected kernel.

In [9]:
import azure.ai.ml, azure.identity, mltable, mlflow, pandas as pd
print("Python:", sys.version.split()[0])
print("azure-ai-ml:", azure.ai.ml.__version__)
print("pandas:", pd.__version__)
print("Environment validation: PASSED")

/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.10.0
azure-ai-ml: 1.35.2
pandas: 2.3.3
Environment validation: PASSED


## 5. Workspace connection

**Objective:** Create one canonical MLClient and validate the workspace.

In [10]:
import src.connect_workspace as connect_workspace
importlib.reload(connect_workspace)
ml_client = connect_workspace.get_ml_client()
connect_workspace.validate_workspace(ml_client, WORKSPACE_NAME, RESOURCE_GROUP)
print("Workspace:", ml_client.workspace_name)
print("Resource group:", ml_client.resource_group_name)
print("Workspace connection: PASSED")

Found the config file in: /config.json


Workspace: udacity-project
Resource group: rg-udacity-azureml
Workspace connection: PASSED


## 6. Compute Instance and Compute Cluster validation

**Objective:** Validate existing compute without creating or deleting resources.

In [11]:
inventory = {item.name: item for item in ml_client.compute.list()}
if COMPUTE_INSTANCE_NAME not in inventory:
    raise RuntimeError(f"Compute Instance {COMPUTE_INSTANCE_NAME!r} not found.")
if COMPUTE_NAME not in inventory:
    raise RuntimeError(f"Compute Cluster {COMPUTE_NAME!r} not found.")
compute_instance = ml_client.compute.get(COMPUTE_INSTANCE_NAME)
compute_cluster = ml_client.compute.get(COMPUTE_NAME)
print("Compute Instance:", compute_instance.name, "| State:", getattr(compute_instance, "state", None))
print("Compute Cluster:", compute_cluster.name)
print("Provisioning state:", compute_cluster.provisioning_state)
print("Minimum nodes:", compute_cluster.min_instances)
print("Maximum nodes:", compute_cluster.max_instances)
print("Idle scale-down seconds:", compute_cluster.idle_time_before_scale_down)

cluster_state = str(compute_cluster.provisioning_state).split(".")[-1].lower()

if cluster_state != "succeeded":
    raise RuntimeError(
        f"Cluster is not Succeeded. Current state: {cluster_state}"
    )
if compute_cluster.min_instances != EXPECTED_MIN_NODES: raise ValueError("Minimum nodes must be 0.")
if compute_cluster.max_instances != EXPECTED_MAX_NODES: raise ValueError("Maximum nodes must be 1.")
print("Compute validation: PASSED")

Compute Instance: ci-udacity-project | State: ComputeInstanceState.RUNNING
Compute Cluster: cpu-cluster
Provisioning state: ProvisioningState.SUCCEEDED
Minimum nodes: 0
Maximum nodes: 1
Idle scale-down seconds: 120.0
Compute validation: PASSED


## 7. Registered data-asset inventory

**Objective:** List assets before retrieval or registration.

In [12]:
for asset in sorted(list(ml_client.data.list()), key=lambda x: x.name.lower()):
    print(asset.name, "| Latest:", getattr(asset, "latest_version", None), "| Type:", asset.type)

bankmarketing | Latest: 1 | Type: uri_file
bankmarketing-mltable | Latest: 1 | Type: mltable


## 8. Retrieve bankmarketing:1

**Objective:** Retrieve and validate the immutable raw file asset.

In [13]:
raw_csv_asset = ml_client.data.get(RAW_DATA_NAME, version=RAW_DATA_VERSION)
if str(raw_csv_asset.type).lower() != "uri_file":
    raise ValueError(f"Expected uri_file, found {raw_csv_asset.type}")
print("Raw asset:", raw_csv_asset.name, raw_csv_asset.version, raw_csv_asset.type)

Raw asset: bankmarketing 1 uri_file


## 9. Create or refresh the local MLTable definition

**Objective:** Compare the local MLTable file with the expected raw asset path.

In [14]:
import src.prepare_data as prepare_data
importlib.reload(prepare_data)
mltable_folder = repo_root / "data" / "bankmarketing_mltable"
mltable_file = mltable_folder / "MLTable"
expected_text = prepare_data.build_mltable_text(str(raw_csv_asset.path))
mltable_status = prepare_data.ensure_local_mltable_definition(mltable_file, expected_text, ALLOW_LOCAL_MLTABLE_REFRESH)
print("MLTable file:", mltable_file.relative_to(repo_root))
print("Status:", mltable_status)
if mltable_status == "DIFFERENT":
    raise RuntimeError("Local MLTable differs. Review before enabling refresh or registering a new version.")

MLTable file: data/bankmarketing_mltable/MLTable
Status: MATCHED


## 10. Validate the MLTable with pandas

**Objective:** Load the MLTable and confirm the target and basic data quality.

In [15]:
validated_df = prepare_data.load_and_validate_mltable(mltable_folder, TARGET_COLUMN)
print("Rows:", len(validated_df))
print("Columns:", len(validated_df.columns))
print("Duplicate rows:", int(validated_df.duplicated().sum()))
print("Missing cells:", int(validated_df.isna().sum().sum()))
print(validated_df[TARGET_COLUMN].value_counts(dropna=False))
display(validated_df.head())

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Rows: 32950
Columns: 21
Duplicate rows: 9
Missing cells: 0
y
no     29258
yes     3692
Name: count, dtype: int64


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,57,technician,married,high.school,no,no,yes,cellular,may,mon,...,1,999,1,failure,-1.8,92.893,-46.2,1.299,5099.1,no
1,55,unknown,married,unknown,unknown,yes,no,telephone,may,thu,...,2,999,0,nonexistent,1.1,93.994,-36.4,4.860,5191.0,no
2,33,blue-collar,married,basic.9y,no,no,no,cellular,may,fri,...,1,999,1,failure,-1.8,92.893,-46.2,1.313,5099.1,no
3,36,admin.,married,high.school,no,no,no,telephone,jun,fri,...,4,999,0,nonexistent,1.4,94.465,-41.8,4.967,5228.1,no
4,27,housemaid,married,high.school,no,yes,no,cellular,jul,fri,...,2,999,0,nonexistent,1.4,93.918,-42.7,4.963,5228.1,no


## 11. Retrieve bankmarketing-mltable:1 if it exists

**Objective:** Search the separate MLTable asset family before registration.

In [16]:
from azure.core.exceptions import ResourceNotFoundError
try:
    training_data_asset = ml_client.data.get(MLTABLE_NAME, version=MLTABLE_VERSION)
except ResourceNotFoundError:
    training_data_asset = None
print("MLTable exists:", training_data_asset is not None)
if training_data_asset and str(training_data_asset.type).lower() != "mltable":
    raise ValueError(f"Expected mltable, found {training_data_asset.type}")

MLTable exists: True


## 12. Create bankmarketing-mltable:1 only when missing

**Objective:** Register only after search confirms missing and the switch is enabled.

In [17]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
if training_data_asset is None:
    if not ALLOW_MLTABLE_REGISTRATION:
        raise RuntimeError("MLTable asset missing. Set ALLOW_MLTABLE_REGISTRATION=True only after review.")
    candidate = Data(name=MLTABLE_NAME, version=MLTABLE_VERSION, type=AssetTypes.MLTABLE, path=str(mltable_folder),
                     description="Bank Marketing MLTable for Udacity Project 2 SDK v2",
                     tags={"project":"udacity-project-2", "sdk":"v2", "source_asset":"bankmarketing:1"})
    training_data_asset = ml_client.data.create_or_update(candidate)
    print("Created new MLTable asset.")
else:
    print("Reusing existing MLTable asset.")
print(training_data_asset.name, training_data_asset.version, training_data_asset.type)

Reusing existing MLTable asset.
bankmarketing-mltable 1 mltable


## 13. Validate src/settings.py

**Objective:** Confirm shared settings match the confirmed resource state.

In [18]:
import src.settings as settings
importlib.reload(settings)
expected = {"RESOURCE_GROUP":RESOURCE_GROUP,"WORKSPACE_NAME":WORKSPACE_NAME,"COMPUTE_INSTANCE_NAME":COMPUTE_INSTANCE_NAME,
            "COMPUTE_NAME":COMPUTE_NAME,"RAW_DATA_NAME":RAW_DATA_NAME,"RAW_DATA_VERSION":RAW_DATA_VERSION,
            "DATA_NAME":MLTABLE_NAME,"DATA_VERSION":MLTABLE_VERSION,"TARGET_COLUMN":TARGET_COLUMN}
mismatches = {k:{"expected":v,"actual":getattr(settings,k,None)} for k,v in expected.items() if getattr(settings,k,None) != v}
if mismatches: raise ValueError(f"settings.py mismatch: {mismatches}")
print("settings.py validation: PASSED")

settings.py validation: PASSED


## 14. Save Part 0 workflow state

**Objective:** Save a non-secret handoff record for Part 1.

In [19]:
import json
from datetime import datetime, timezone
state_dir = repo_root / "outputs"; state_dir.mkdir(parents=True, exist_ok=True)
part0_state = {"saved_utc":datetime.now(timezone.utc).isoformat(),"workspace":WORKSPACE_NAME,"resource_group":RESOURCE_GROUP,
               "compute_instance":COMPUTE_INSTANCE_NAME,"compute_cluster":COMPUTE_NAME,"raw_data_asset":f"{raw_csv_asset.name}:{raw_csv_asset.version}",
               "training_data_asset":f"{training_data_asset.name}:{training_data_asset.version}","target_column":TARGET_COLUMN,
               "validated_rows":int(len(validated_df)),"validated_columns":int(len(validated_df.columns)),"part0_status":"COMPLETED"}
(repo_root / "outputs" / "part0_workflow_state.json").write_text(json.dumps(part0_state, indent=2), encoding="utf-8")
print(json.dumps(part0_state, indent=2))

{
  "saved_utc": "2026-11-20T21:08:08.462063+00:00",
  "workspace": "udacity-project",
  "resource_group": "rg-udacity-azureml",
  "compute_instance": "ci-udacity-project",
  "compute_cluster": "cpu-cluster",
  "raw_data_asset": "bankmarketing:1",
  "training_data_asset": "bankmarketing-mltable:1",
  "target_column": "y",
  "validated_rows": 32950,
  "validated_columns": 21,
  "part0_status": "COMPLETED"
}


# PART 1 — AUTOML AND MANAGED ONLINE ENDPOINT

## 15. Part 0 handoff validation

**Objective:** Rebuild required objects after restart and validate Part 0 state.

In [20]:
part0_file = repo_root / "outputs" / "part0_workflow_state.json"
if not part0_file.exists(): raise FileNotFoundError("Part 0 state missing.")
part0_state = json.loads(part0_file.read_text())
if part0_state.get("part0_status") != "COMPLETED": raise RuntimeError("Part 0 is not complete.")
training_data_asset = ml_client.data.get(settings.DATA_NAME, version=settings.DATA_VERSION)
validated_df = prepare_data.load_and_validate_mltable(mltable_folder, settings.TARGET_COLUMN)
print("Part 0 handoff validation: PASSED")

Part 0 handoff validation: PASSED


## 16. Data-quality summary

**Objective:** Print evaluator-facing training data checks.

In [21]:
dq_summary={"rows":len(validated_df),"columns":len(validated_df.columns),"duplicates":int(validated_df.duplicated().sum()),
            "missing_cells":int(validated_df.isna().sum().sum()),"target_values":validated_df[settings.TARGET_COLUMN].value_counts(dropna=False).to_dict()}
print(dq_summary)

{'rows': 32950, 'columns': 21, 'duplicates': 9, 'missing_cells': 0, 'target_values': {'no': 29258, 'yes': 3692}}


## 17–20. Search, reuse, or explicitly submit AutoML

**Objective:** Reuse the newest suitable Completed job; submit only when allowed.

In [22]:
import src.run_automl as automl_workflow
importlib.reload(automl_workflow)
selected_automl_job, submitted_new_job = automl_workflow.get_or_submit_automl_job(
    ml_client=ml_client, experiment_name=settings.EXPERIMENT_NAME, training_data_asset=training_data_asset,
    compute_name=settings.COMPUTE_NAME, target_column=settings.TARGET_COLUMN, allow_new_job=ALLOW_NEW_AUTOML_JOB,
    max_trials=settings.MAX_TRIALS, max_concurrent_trials=settings.MAX_CONCURRENT_TRIALS)
print("Selected parent:", selected_automl_job.name, "| Status:", selected_automl_job.status)
print("Submitted now:", submitted_new_job)
print("Studio URL:", getattr(selected_automl_job, "studio_url", None))
if submitted_new_job:
    ml_client.jobs.stream(selected_automl_job.name)
    selected_automl_job = ml_client.jobs.get(selected_automl_job.name)
    if selected_automl_job.status != "Completed": raise RuntimeError(f"AutoML ended {selected_automl_job.status}")

Selected parent: quirky_moon_6pbsp2k6rn | Status: JobStatus.COMPLETED
Submitted now: False
Studio URL: https://ml.azure.com/runs/quirky_moon_6pbsp2k6rn?wsid=/subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourcegroups/rg-udacity-azureml/workspaces/udacity-project


In [23]:
# import importlib
# import src.run_automl as automl_workflow

# importlib.reload(automl_workflow)

# # Recheck existing completed jobs
# candidates = automl_workflow.find_suitable_completed_jobs(
#     ml_client=ml_client,
#     experiment_name=settings.EXPERIMENT_NAME,
#     data_asset=training_data_asset,
#     compute_name=settings.COMPUTE_NAME,
#     target_column=settings.TARGET_COLUMN
# )

# print("Suitable completed jobs:", len(candidates))

# for job in candidates:
#     print(
#         "Job:", job.name,
#         "| Status:", job.status
#     )

## 21–24. Review child jobs and validate MLflow artifact

**Objective:** Identify the winning Completed child and find its MLmodel artifact.

In [24]:
import importlib
import model_inspection.inspection as inspection

importlib.reload(inspection)

# ============================================================
# 1. INSPECT AUTOML PARENT JOB
# ============================================================
print("=" * 65)
print("AUTOML PARENT JOB")
print("=" * 65)

inspection.inspect_parent_properties(selected_automl_job)

# ============================================================
# 2. LIST CHILD JOBS
# ============================================================
child_jobs = inspection.list_child_jobs(
    ml_client,
    selected_automl_job.name
)

inspection.print_child_job_summary(child_jobs)

# ============================================================
# 3. MANUAL FALLBACK
# ============================================================
BEST_CHILD_JOB_NAME_OVERRIDE = "quirky_moon_6pbsp2k6rn_3"

child_job_names = {job.name for job in child_jobs}

if BEST_CHILD_JOB_NAME_OVERRIDE not in child_job_names:
    raise ValueError(
        f"Winning child job not found: {BEST_CHILD_JOB_NAME_OVERRIDE}"
    )

best_child_job = ml_client.jobs.get(
    name=BEST_CHILD_JOB_NAME_OVERRIDE
)

# ============================================================
# 4. VALIDATE AND DISPLAY
# ============================================================
if inspection._normalized_status(best_child_job.status) != "completed":
    raise RuntimeError("Winning child job is not completed.")

print("\n" + "=" * 65)
print("SELECTED BEST MODEL")
print("=" * 65)

print("Parent job:", selected_automl_job.name)
print("Best model display name:", best_child_job.display_name)
print("Best model child job:", best_child_job.name)
print("Status:", best_child_job.status)

print("\nBest child job is ready for registration.")


AUTOML PARENT JOB
PARENT JOB PROPERTIES
num_iterations: 5
training_type: TrainFull
acquisition_function: EI
primary_metric: accuracy
train_split: 0
acquisition_parameter: 0
num_cross_validation: 5
target: cpu-cluster
AMLSettingsJsonString: {"is_subgraph_orchestration":false,"is_automode":true,"path":"./sample_projects/bankmarketing-automl-sdk-v2","subscription_id":"cf903865-71df-48c8-9cb3-1938981c8c3a","resource_group":"rg-udacity-azureml","workspace_name":"udacity-project","compute_target":"cpu-cluster","iterations":5,"primary_metric":"accuracy","task_type":"classification","IsImageTask":false,"IsTextDNNTask":false,"n_cross_validations":5,"preprocess":true,"is_timeseries":false,"time_column_name":null,"grain_column_names":null,"max_cores_per_iteration":-1,"max_concurrent_iterations":1,"max_nodes":1,"iteration_timeout_minutes":20,"enforce_time_on_windows":false,"experiment_timeout_minutes":60,"exit_score":"NaN","experiment_exit_score":"NaN","auto_blacklist":false,"blacklist_samples_rea

## 25–27. Reuse or register the MLflow model

**Objective:** Reuse the model with matching source_job lineage.

In [25]:
print("Best child job:", best_child_job.name)
print("Display name:", best_child_job.display_name)
print("Status:", best_child_job.status)

Best child job: quirky_moon_6pbsp2k6rn_3
Display name: mighty_fennel_qh0gnpwy
Status: JobStatus.COMPLETED


In [26]:
# from importlib.metadata import version

# for package in [
#     "azure-ai-ml",
#     "azureml-mlflow",
#     "mlflow",
#     "mlflow-skinny",
# ]:
#     print(f"{package}: {version(package)}")

In [27]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

run = client.get_run("quirky_moon_6pbsp2k6rn_3")

print("Run ID:", run.info.run_id)
print("Artifact URI:", run.info.artifact_uri)

artifacts = client.list_artifacts(run.info.run_id)

for artifact in artifacts:
    print(
        "DIRECTORY" if artifact.is_dir else "FILE",
        artifact.path
    )

Run ID: quirky_moon_6pbsp2k6rn_3
Artifact URI: azureml://eastus.api.azureml.ms/mlflow/v2.0/subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourceGroups/rg-udacity-azureml/providers/Microsoft.MachineLearningServices/workspaces/udacity-project/experiments/688871eb-0731-42aa-be04-3aace6778f5b/runs/quirky_moon_6pbsp2k6rn_3/artifacts
FILE accuracy_table
FILE automl_driver.py
FILE confusion_matrix
DIRECTORY explanation
DIRECTORY logs
DIRECTORY outputs
DIRECTORY system_logs
DIRECTORY user_logs


In [28]:
# import importlib
# import src.register_best_model as model_registration

# importlib.reload(model_registration)

# # Retrieve the confirmed winning child job
# best_child_job = ml_client.jobs.get(
#     name="quirky_moon_6pbsp2k6rn_3"
# )

# # Register the winning model
# registered_model, was_created = (
#     model_registration.get_or_register_model(
#         ml_client=ml_client,
#         model_name="bankmarketing-automl-model",
#         best_child_job=best_child_job,
#         allow_registration=True,
#     )
# )

# print("\n" + "=" * 60)
# print("MODEL REGISTRATION RESULT")
# print("=" * 60)
# print("Model name:", registered_model.name)
# print("Model version:", registered_model.version)
# print("New registration:", was_created)

## Step  — Register the Best AutoML Model in Azure ML Studio

### Why manual registration?

The winning AutoML child job has been identified, but automated artifact discovery and registration encountered an incompatibility between the Azure ML MLflow tracking backend and MLflow 3.x.

We will register the winning model directly through Azure Machine Learning Studio. This avoids repeating AutoML training and preserves the original SDK v2 workflow.

### Winning model details

| Property | Value |
|---|---|
| Parent AutoML job | `quirky_moon_6pbsp2k6rn` |
| Winning child job | `quirky_moon_6pbsp2k6rn_3` |
| Winning model display name | `mighty_fennel_qh0gnpwy` |
| Model registry name | `bankmarketing-automl-model` |
| Model type | MLflow |

### Manual registration procedure

1. Open [Azure Machine Learning Studio](https://ml.azure.com/).
2. Select the `udacity-project` workspace.
3. Navigate to **Jobs**.
4. Open the parent AutoML job `quirky_moon_6pbsp2k6rn`.
5. Navigate to **Models + child jobs**, if available.
6. Locate the winning model `mighty_fennel_qh0gnpwy`, corresponding to child job `quirky_moon_6pbsp2k6rn_3`.
7. Open the winning child job and navigate to **Outputs + logs**.
8. Locate the `default` output and browse its contents.
9. Expand the `outputs` directory and locate the actual model folder containing the `MLmodel` file.
10. Select **Register model**.
11. Choose **MLflow** as the model type.
12. Select the folder containing `MLmodel` as the model source.
13. Enter the following model settings:

   - Name: `bankmarketing-automl-model`
   - Description: Best Bank Marketing AutoML model for Udacity Project 2
14. Review the configuration and select **Register**.
15. Wait for registration to complete successfully.

### Important validation

After registration, navigate to **Models** in Azure ML Studio.

Confirm that `bankmarketing-automl-model` appears in the model registry and record its registered version.

The model must be registered as an **MLflow model**, not as an unspecified or custom model.

Do not proceed to deployment until registration is confirmed.


In [29]:

# ============================================================
# VALIDATE MANUALLY REGISTERED MODEL
# ============================================================

from azure.core.exceptions import ResourceNotFoundError

REGISTERED_MODEL_NAME = "bankmarketing-automl-model"

# Update this value using the version shown in Azure ML Studio.
REGISTERED_MODEL_VERSION = "1"

print("=" * 65)
print("REGISTERED MODEL VALIDATION")
print("=" * 65)

try:
    registered_model = ml_client.models.get(
        name=REGISTERED_MODEL_NAME,
        version=REGISTERED_MODEL_VERSION,
    )

except ResourceNotFoundError as exc:
    raise RuntimeError(
        f"Model {REGISTERED_MODEL_NAME}, version "
        f"{REGISTERED_MODEL_VERSION}, was not found. "
        "Verify the model name and version in Azure ML Studio."
    ) from exc

# Validate model identity
assert registered_model.name == REGISTERED_MODEL_NAME

# Validate model type
model_type = str(registered_model.type).lower()

if "mlflow" not in model_type:
    raise ValueError(
        f"Expected an MLflow model, but received: {registered_model.type}"
    )

print("Model name:", registered_model.name)
print("Model version:", registered_model.version)
print("Model type:", registered_model.type)
print("Model path:", registered_model.path)
print("Model description:", registered_model.description)

print("\nModel registration validation PASSED.")


REGISTERED MODEL VALIDATION
Model name: bankmarketing-automl-model
Model version: 1
Model type: mlflow_model
Model path: azureml://subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourceGroups/rg-udacity-azureml/workspaces/udacity-project/datastores/workspaceartifactstore/paths/ExperimentRun/dcid.quirky_moon_6pbsp2k6rn_3/outputs/mlflow-model
Model description: Best Bank Marketing AutoML model for Udacity Project 2

Model registration validation PASSED.


## 28–29. Build and validate scoring request

**Objective:** Create a safe request from validated_df and exclude target y.

In [30]:

# ============================================================
# BUILD AND VALIDATE DEPLOYMENT SAMPLE REQUEST
# ============================================================

import importlib
import deployment.request_builder as request_builder

importlib.reload(request_builder)

request_payload, request_file = request_builder.build_and_save_request(
    validated_df,
    settings.TARGET_COLUMN,
    repo_root / "deployment" / "sample-request.json",
    row_count=2,
)

request_builder.validate_request(
    request_payload,
    [settings.TARGET_COLUMN],
)

print("Request file:", request_file.relative_to(repo_root))
print(
    "Target excluded:",
    settings.TARGET_COLUMN not in request_payload["input_data"]["columns"],
)


Request file: deployment/sample-request.json
Target excluded: True


In [31]:
import importlib
import src.settings as settings

importlib.reload(settings)

import src.deploy_online as online_deployment
importlib.reload(online_deployment)

print("Endpoint:", settings.ONLINE_ENDPOINT_NAME)
print("Deployment:", settings.DEPLOYMENT_NAME)
print("Instance type:", settings.ONLINE_INSTANCE_TYPE)
print("Instance count:", settings.ONLINE_INSTANCE_COUNT)

Endpoint: bankmarketing-endpoint
Deployment: blue
Instance type: Standard_DS1_v2
Instance count: 1


## 30–32. Retrieve or create endpoint and blue deployment

**Objective:** Search, validate, and conditionally create online resources.

In [32]:
import importlib
import src.deploy_online as online_deployment

importlib.reload(online_deployment)

print("Deployment module reloaded successfully.")

Deployment module reloaded successfully.


In [33]:
# ============================================================
# CELL 33: CHECK ENDPOINT STATUS AND DELETE IF FAILED
# ============================================================

from azure.core.exceptions import ResourceNotFoundError

endpoint_name = settings.ONLINE_ENDPOINT_NAME

# Track deletion activity for Cell 34.
endpoint_deletion_requested = False

try:
    existing_endpoint = ml_client.online_endpoints.get(
        name=endpoint_name
    )

    endpoint_state = str(
        existing_endpoint.provisioning_state
    ).split(".")[-1].lower()

    print("Endpoint:", existing_endpoint.name)
    print("Provisioning state:", endpoint_state)

    if endpoint_state == "succeeded":

        print("Endpoint is healthy. No action required.")
        print("Endpoint will be retained.")

    elif endpoint_state == "failed":

        print("Endpoint provisioning failed.")
        print("Deleting failed endpoint...")

        ml_client.online_endpoints.begin_delete(
            name=endpoint_name
        ).result()

        endpoint_deletion_requested = True

        print("Endpoint deletion operation completed.")

    else:

        print(
            f"Endpoint is in '{endpoint_state}' state. "
            "No deletion performed."
        )

except ResourceNotFoundError:

    print("Endpoint does not exist. No action required.")

Endpoint: bankmarketing-endpoint
Provisioning state: succeeded
Endpoint is healthy. No action required.
Endpoint will be retained.


In [34]:
# ============================================================
# CELL 34: CONFIRM ENDPOINT DELETION
# ============================================================

from azure.core.exceptions import ResourceNotFoundError

endpoint_name = settings.ONLINE_ENDPOINT_NAME

try:
    endpoint_check = ml_client.online_endpoints.get(
        name=endpoint_name
    )

    print("Endpoint still exists.")
    print("Endpoint:", endpoint_check.name)
    print("Provisioning state:", endpoint_check.provisioning_state)

    if endpoint_deletion_requested:
        raise RuntimeError(
            "Endpoint deletion was requested, "
            "but the endpoint still exists."
        )

    print("No deletion confirmation required.")

except ResourceNotFoundError:

    if endpoint_deletion_requested:
        print("SUCCESS: Endpoint deletion confirmed.")
        print("Deleted endpoint:", endpoint_name)

    else:
        print("Endpoint does not exist.")
        print("No deletion was performed in Cell 33.")

Endpoint still exists.
Endpoint: bankmarketing-endpoint
Provisioning state: EndpointProvisioningState.SUCCEEDED
No deletion confirmation required.


In [35]:
# Recreate only the endpoint
ALLOW_ENDPOINT_CREATION = True
ALLOW_DEPLOYMENT_CREATION = False
ALLOW_TRAFFIC_UPDATE = False

# Reload the modules
import importlib
import src.settings as settings
import src.deploy_online as online_deployment

importlib.reload(settings)
importlib.reload(online_deployment)

print("Endpoint:", settings.ONLINE_ENDPOINT_NAME)
print("Deployment VM:", settings.ONLINE_INSTANCE_TYPE)
print("Deployment instances:", settings.ONLINE_INSTANCE_COUNT)

Endpoint: bankmarketing-endpoint
Deployment VM: Standard_DS1_v2
Deployment instances: 1


In [36]:
# create the endpoint
endpoint = online_deployment.get_or_create_endpoint(
	ml_client=ml_client,
	endpoint_name=settings.ONLINE_ENDPOINT_NAME,
	allow_creation=ALLOW_ENDPOINT_CREATION,
)
print("Endpoint:", endpoint.name)
print("Provisioning state:", endpoint.provisioning_state)
print("Traffic:", endpoint.traffic)
print("Scoring URI:", endpoint.scoring_uri)


Endpoint: bankmarketing-endpoint
Provisioning state: EndpointProvisioningState.SUCCEEDED
Traffic: {'blue': 100}
Scoring URI: https://bankmarketing-endpoint.eastus.inference.ml.azure.com/score


In [37]:
# Do not create blue unless the endpoint succeeded
endpoint_state = str(
	endpoint.provisioning_state
).split(".")[-1].lower()
if endpoint_state != "succeeded":
	raise RuntimeError(
    	f"Endpoint {endpoint.name!r} is not ready. "
    	f"Current state: {endpoint.provisioning_state}. "
    	"Do not create the deployment."
	)
print("Endpoint readiness validation: PASSED")

Endpoint readiness validation: PASSED


In [38]:
# Create blue with Standard_DS1_v2
ALLOW_ENDPOINT_CREATION = False
ALLOW_DEPLOYMENT_CREATION = True
ALLOW_TRAFFIC_UPDATE = False

deployment = online_deployment.get_or_create_deployment(
	ml_client=ml_client,
	endpoint_name=endpoint.name,
	deployment_name=settings.DEPLOYMENT_NAME,
	model=registered_model,
	allow_creation=ALLOW_DEPLOYMENT_CREATION,
)
print("Deployment:", deployment.name)
print("Provisioning state:", deployment.provisioning_state)
print("VM size:", deployment.instance_type)
print("Instance count:", deployment.instance_count)
print("Application Insights:", deployment.app_insights_enabled)


Deployment: blue
Provisioning state: DeploymentProvisioningState.SUCCEEDED
VM size: Standard_DS1_v2
Instance count: 1
Application Insights: True


In [39]:
# Route traffic only after blue succeeds
deployment_state = str(
	deployment.provisioning_state
).split(".")[-1].lower()
if deployment_state != "succeeded":
	raise RuntimeError(
    	f"Deployment is not ready: {deployment.provisioning_state}"
	)
ALLOW_TRAFFIC_UPDATE = True
endpoint = online_deployment.ensure_traffic(
	ml_client=ml_client,
	endpoint=endpoint,
	deployment_name=deployment.name,
	allow_update=ALLOW_TRAFFIC_UPDATE,
)
print("Traffic:", endpoint.traffic)
print("Scoring URI:", endpoint.scoring_uri)


Traffic: {'blue': 100}
Scoring URI: https://bankmarketing-endpoint.eastus.inference.ml.azure.com/score


## 33. Retrieve deployment logs

**Objective:** Get Managed Online Deployment logs through MLClient.

In [40]:
import deployment.logging as deployment_logging
importlib.reload(deployment_logging)
log_text = deployment_logging.get_deployment_logs(ml_client, endpoint.name, deployment.name, lines=100)
print(log_text)

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-11-17T05:12:39.478731Z, Message: Start pulling container image
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-11-17T05:12:40.173713Z, Message: Start downloading models
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-11-17T05:15:26.58608Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-11-17T05:15:26.58608Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-11-17T05:15:26.642156Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-11-17T05:15:26.711373Z, Message: Started container inference-server
Kind: Pod, Name: LivenessProbeFailed, Type: Warning, Time: 2026-11-17T05:15:47.984958Z, Message: Liveness probe failed: Get \"http://10.66.0.2:50

## 34. Invoke endpoint and validate prediction

**Objective:** Invoke with the saved JSON request without printing keys.

In [41]:
import deployment.scoring as scoring
importlib.reload(scoring)
prediction = scoring.invoke_and_validate(ml_client, endpoint.name, request_file, deployment.name)
print("Prediction:", prediction)

Prediction: ['no', 'no']


## 35–36. Monitoring traffic and optional benchmark

**Objective:** Generate controlled evidence traffic and optional latency summary.

In [42]:
import deployment.monitoring as monitoring
importlib.reload(monitoring)
print(monitoring.generate_monitoring_requests(ml_client, endpoint.name, deployment.name, request_file, 10, ALLOW_MONITORING_REQUESTS))
print(monitoring.run_lightweight_benchmark(ml_client, endpoint.name, deployment.name, request_file, 5, ALLOW_BENCHMARK))

{'status': 'COMPLETED', 'requests': 10, 'successful_responses': 10}
{'status': 'SKIPPED', 'reason': 'Benchmark is disabled.'}


## 37. Save Part 1 workflow state

**Objective:** Save selected resource names and versions without secrets.

In [43]:
part1_state={"saved_utc":datetime.now(timezone.utc).isoformat(),"part1_status":"COMPLETED","automl_parent_job":selected_automl_job.name,
             "best_child_job":best_child_job.name,"model_name":registered_model.name,"model_version":str(registered_model.version),
             "online_endpoint":endpoint.name,"online_deployment":deployment.name}
(repo_root / "outputs" / "part1_workflow_state.json").write_text(json.dumps(part1_state, indent=2), encoding="utf-8")
print(json.dumps(part1_state, indent=2))

{
  "saved_utc": "2026-11-20T21:08:42.781145+00:00",
  "part1_status": "COMPLETED",
  "automl_parent_job": "quirky_moon_6pbsp2k6rn",
  "best_child_job": "quirky_moon_6pbsp2k6rn_3",
  "model_name": "bankmarketing-automl-model",
  "model_version": "1",
  "online_endpoint": "bankmarketing-endpoint",
  "online_deployment": "blue"
}


## 38. Validate Part 1 screenshot inventory

**Objective:** Check the evaluator evidence filenames.

In [44]:
expected_part1=["01_mltable_asset_preview.png","02_automl_parent_job_completed.png","03_automl_models_and_child_jobs.png",
"04_winning_child_job.png","05_mlflow_mlmodel_artifact.png","06_registered_model_lineage.png","07_managed_online_endpoint.png",
"08_blue_deployment_app_insights.png","09_deployment_logs.png","10_successful_prediction.png","11_endpoint_monitoring_traffic.png","12_part1_final_state.png"]
for name in expected_part1:
    p=repo_root / "screenshots" / "part1" / name
    print("FOUND" if p.exists() else "MISSING", p.relative_to(repo_root))

MISSING screenshots/part1/01_mltable_asset_preview.png
MISSING screenshots/part1/02_automl_parent_job_completed.png
MISSING screenshots/part1/03_automl_models_and_child_jobs.png
MISSING screenshots/part1/04_winning_child_job.png
MISSING screenshots/part1/05_mlflow_mlmodel_artifact.png
MISSING screenshots/part1/06_registered_model_lineage.png
MISSING screenshots/part1/07_managed_online_endpoint.png
MISSING screenshots/part1/08_blue_deployment_app_insights.png
MISSING screenshots/part1/09_deployment_logs.png
MISSING screenshots/part1/10_successful_prediction.png
MISSING screenshots/part1/11_endpoint_monitoring_traffic.png
MISSING screenshots/part1/12_part1_final_state.png


## 39. Part 1 cleanup gate

**Objective:** Keep cleanup disabled until screenshots and screencast are secured.

In [45]:
if ALLOW_CLEANUP:
    print("Cleanup enabled. Place reviewed delete calls in separate visible cells only.")
else:
    print("Cleanup skipped. ALLOW_CLEANUP is False.")

Cleanup skipped. ALLOW_CLEANUP is False.


# PART 2 — PIPELINE OPERATIONALIZATION

## 40. Part 1 handoff validation

**Objective:** Load Part 1 state and selected model after restart.

In [46]:
part1_file=repo_root/"outputs"/"part1_workflow_state.json"
if not part1_file.exists(): raise FileNotFoundError("Part 1 state missing.")
part1_state=json.loads(part1_file.read_text())
if part1_state.get("part1_status") != "COMPLETED": raise RuntimeError("Part 1 is not complete.")
registered_model=ml_client.models.get(part1_state["model_name"],version=part1_state["model_version"])
print("Part 1 handoff validation: PASSED")

Part 1 handoff validation: PASSED


In [47]:
# ============================================================
# PART 2: REBUILD REQUIRED CONTEXT
# ============================================================
import importlib
import src.settings as settings
import pipeline.pipeline_definition as pipeline_definition

importlib.reload(settings)
importlib.reload(pipeline_definition)

# Retrieve the registered training data asset.
training_data_asset = ml_client.data.get(
    name=settings.DATA_NAME,
    version=settings.DATA_VERSION,
)

# Retrieve the registered model from the Part 1 handoff.
part1_file = repo_root / "outputs" / "part1_workflow_state.json"
part1_state = json.loads(part1_file.read_text(encoding="utf-8"))

registered_model = ml_client.models.get(
    name=part1_state["model_name"],
    version=part1_state["model_version"],
)

print("Settings module:", settings.__file__)
print("Pipeline experiment:", settings.PIPELINE_EXPERIMENT_NAME)
print("Compute:", settings.COMPUTE_NAME)
print("Target:", settings.TARGET_COLUMN)
print("Training data:", training_data_asset.name)
print("Training data version:", training_data_asset.version)
print("Registered model:", registered_model.name, registered_model.version)

Settings module: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/src/settings.py
Pipeline experiment: bankmarketing-pipeline-sdk-v2
Compute: cpu-cluster
Target: y
Training data: bankmarketing-mltable
Training data version: 1
Registered model: bankmarketing-automl-model 1


## 41. Build or import SDK v2 pipeline definition

**Objective:** Build the graph without submitting it.

In [48]:
# ============================================================
# BUILD PIPELINE DEFINITION
# ============================================================

pipeline_job_definition = pipeline_definition.build_training_pipeline(
    f"azureml:{training_data_asset.name}:{training_data_asset.version}",
    settings.COMPUTE_NAME,
    settings.TARGET_COLUMN,
)

print("Pipeline definition built successfully.")
print("No Azure job submitted.")

Pipeline definition built successfully.
No Azure job submitted.


In [49]:
from pathlib import Path

# Align each component's packaging base path with its source directory
for step_name in ["validate_step", "train_step"]:

    component = pipeline_job_definition.jobs[step_name].component

    component._base_path = Path(component.code).resolve()

    print(f"{step_name}:")
    print(f"  Code:      {component.code}")
    print(f"  Base path: {component._base_path}")

validate_step:
  Code:      /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/pipeline/steps
  Base path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/pipeline/steps
train_step:
  Code:      /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/pipeline/steps
  Base path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/pipeline/steps


## 42–45. Search, reuse, or submit pipeline job

**Objective:** Reuse a matching Completed pipeline job or submit only under the explicit switch.

In [50]:
# ============================================================
# FIND OR SUBMIT PIPELINE JOB — SAFE REUSE GATE
# ============================================================

# Safety switch: Never submit another job unless explicitly enabled.
ALLOW_NEW_PIPELINE_JOB = False

# Explicitly select the known successful pipeline job.
PREFERRED_PIPELINE_JOB_NAME = (
    "pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913"
)

# ============================================================
# STEP 1: INSPECT EXISTING PIPELINE JOBS
# ============================================================

existing_jobs = list(ml_client.jobs.list())

experiment_jobs = [
    job
    for job in existing_jobs
    if getattr(job, "experiment_name", None)
    == settings.PIPELINE_EXPERIMENT_NAME
    and getattr(job, "type", None) == "pipeline"
]

print("Pipeline experiment:", settings.PIPELINE_EXPERIMENT_NAME)
print("Pipeline jobs found:", len(experiment_jobs))

for job in experiment_jobs:
    print(
        "Job:", job.name,
        "| Status:", job.status,
        "| Type:", getattr(job, "type", None),
    )

# ============================================================
# STEP 2: IDENTIFY COMPLETED PIPELINE JOBS
# ============================================================

completed_pipeline_jobs = [
    job
    for job in experiment_jobs
    if str(job.status).split(".")[-1].lower() == "completed"
]

print("\nCompleted pipeline jobs:", len(completed_pipeline_jobs))

# ============================================================
# STEP 3: SELECT THE INTENDED EXISTING JOB
# ============================================================

completed_job_names = [
    job.name for job in completed_pipeline_jobs
]

if PREFERRED_PIPELINE_JOB_NAME in completed_job_names:

    completed_pipeline_job = ml_client.jobs.get(
        PREFERRED_PIPELINE_JOB_NAME
    )

    print("\nReusing explicitly selected pipeline:")
    print(completed_pipeline_job.name)

elif len(completed_pipeline_jobs) == 1:

    completed_pipeline_job = ml_client.jobs.get(
        completed_pipeline_jobs[0].name
    )

    print("\nReusing the only completed pipeline:")
    print(completed_pipeline_job.name)

elif len(completed_pipeline_jobs) > 1:

    raise RuntimeError(
        "Multiple completed jobs exist, but the preferred job "
        "was not found. Review the job names before proceeding."
    )

else:

    completed_pipeline_job = None
    print("\nNo completed pipeline found.")

# ============================================================
# STEP 4: SUBMIT ONLY IF NO SUCCESSFUL JOB EXISTS
# ============================================================

submitted_pipeline_job = None

if completed_pipeline_job is None:

    if not ALLOW_NEW_PIPELINE_JOB:
        raise RuntimeError(
            "No successful pipeline job exists in the target "
            "experiment. New pipeline submission is disabled. "
            "No new job was submitted."
        )

    print("Submitting a new pipeline job...")

    EXPECTED_PIPELINE_TAGS = {
        "project": "udacity-project-2",
        "sdk": "v2",
        "data_asset": (
            f"{training_data_asset.name}:"
            f"{training_data_asset.version}"
        ),
        "target_column": settings.TARGET_COLUMN,
        "compute": settings.COMPUTE_NAME,
        "purpose": "pipeline-operationalization",
    }

    pipeline_job_definition.tags = EXPECTED_PIPELINE_TAGS

    submitted_pipeline_job = ml_client.jobs.create_or_update(
        pipeline_job_definition,
        experiment_name=settings.PIPELINE_EXPERIMENT_NAME,
    )

    print("Submitted pipeline job:", submitted_pipeline_job.name)
    ml_client.jobs.stream(submitted_pipeline_job.name)

    completed_pipeline_job = ml_client.jobs.get(
        submitted_pipeline_job.name
    )

# ============================================================
# STEP 5: VALIDATE COMPLETED PIPELINE
# ============================================================

if str(completed_pipeline_job.status).split(".")[-1].lower() != "completed":
    raise RuntimeError(
        "Pipeline did not complete successfully. "
        f"Status: {completed_pipeline_job.status}"
    )

pipeline_children = (
    pipeline_definition.validate_completed_pipeline_job(
        ml_client,
        completed_pipeline_job,
    )
)

# ============================================================
# STEP 6: FINAL PIPELINE SUMMARY
# ============================================================

print("\n========== PIPELINE JOB SUMMARY ==========")
print("Pipeline job:", completed_pipeline_job.name)
print("Status:", completed_pipeline_job.status)
print("Studio URL:", completed_pipeline_job.studio_url)
print("Child jobs:", len(pipeline_children))
print(
    "New submission:",
    submitted_pipeline_job is not None,
)

print("\nPipeline reuse/submission gate passed.")

Pipeline experiment: bankmarketing-pipeline-sdk-v2
Pipeline jobs found: 3
Job: pipelinejob-f9dfbe05-dc0a-4d98-916a-05800c5f5d24 | Status: JobStatus.COMPLETED | Type: pipeline
Job: pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913 | Status: JobStatus.COMPLETED | Type: pipeline
Job: kind_ring_b8dhsp854d | Status: JobStatus.COMPLETED | Type: pipeline

Completed pipeline jobs: 3

Reusing explicitly selected pipeline:
pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913
Pipeline job: pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913
Pipeline status: completed

========== CHILD JOB STATUS ==========
Name: c104829a-576b-487a-aad7-c7bbd7e63139 | Status: completed
Name: c9164993-11fe-475a-9841-9d69e33e06d0 | Status: completed

All pipeline child jobs completed successfully.

========== PIPELINE JOB SUMMARY ==========
Pipeline job: pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913
Status: JobStatus.COMPLETED
Studio URL: https://ml.azure.com/runs/pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913?wsid=/su

In [51]:
# Pipeline
ALLOW_NEW_PIPELINE_JOB = False

# Component registration
ALLOW_COMPONENT_REGISTRATION = True

# Batch Endpoint and deployment
ALLOW_BATCH_ENDPOINT_CREATION = True
ALLOW_BATCH_DEPLOYMENT_CREATION = True
ALLOW_DEFAULT_DEPLOYMENT_UPDATE = True

# Batch Endpoint invocation
ALLOW_BATCH_INVOCATION = True

# Schedule
ALLOW_SCHEDULE_CREATION = False
ALLOW_SCHEDULE_ENABLE = False

# Cleanup
ALLOW_CLEANUP = False

In [52]:
print("Settings module:", settings.__file__)
print("Current component name:", settings.PIPELINE_COMPONENT_NAME)

Settings module: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/src/settings.py
Current component name: bankmarketing_training_pipeline


In [53]:
PIPELINE_COMPONENT_NAME = "bankmarketing-training-pipeline"

## 46. Retrieve or register reusable pipeline component

**Objective:** Reuse name/version or register only when absent.

In [54]:
import importlib
import src.settings as settings

importlib.reload(settings)

print("Settings module:", settings.__file__)
print("Component name:", settings.PIPELINE_COMPONENT_NAME)
print("Component version:", settings.PIPELINE_COMPONENT_VERSION)

Settings module: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/src/settings.py
Component name: bankmarketing_training_pipeline
Component version: 1


In [55]:
import importlib

import src.settings as settings
import pipeline.pipeline_definition as pipeline_definition
import pipeline.component_registration as component_registration

importlib.reload(settings)
importlib.reload(pipeline_definition)
importlib.reload(component_registration)

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

ALLOW_COMPONENT_REGISTRATION = True
ALLOW_NEW_PIPELINE_JOB = False
ALLOW_BATCH_ENDPOINT_CREATION = False
ALLOW_BATCH_DEPLOYMENT_CREATION = False
ALLOW_DEFAULT_DEPLOYMENT_UPDATE = False
ALLOW_BATCH_INVOCATION = False
ALLOW_SCHEDULE_CREATION = False
ALLOW_SCHEDULE_ENABLE = False
ALLOW_CLEANUP = False

print("Workspace:", ml_client.workspace_name)
print("Component:", settings.PIPELINE_COMPONENT_NAME)
print("Compute:", settings.COMPUTE_NAME)
print("New pipeline jobs:", ALLOW_NEW_PIPELINE_JOB)
print("Batch invocation:", ALLOW_BATCH_INVOCATION)

Workspace: udacity-project
Component: bankmarketing_training_pipeline
Compute: cpu-cluster
New pipeline jobs: False
Batch invocation: False


In [56]:
# ============================================================
# REGISTER PIPELINE COMPONENT
# ============================================================
import pipeline.component_registration as component_registration

importlib.reload(component_registration)

registered_pipeline_component, component_created = (
    component_registration.get_or_register_pipeline_component(
        ml_client=ml_client,
        pipeline_job_definition=pipeline_job_definition,
        component_name=settings.PIPELINE_COMPONENT_NAME,
        component_version=settings.PIPELINE_COMPONENT_VERSION,
        allow_registration=ALLOW_COMPONENT_REGISTRATION,
    )
)

print("\n========== PIPELINE COMPONENT ==========")
print("Name:", registered_pipeline_component.name)
print("Version:", registered_pipeline_component.version)
print("Created:", component_created)
print("Type:", registered_pipeline_component.type)

Reusing existing pipeline component:
Name: bankmarketing_training_pipeline
Version: 1

========== PIPELINE COMPONENT ==========
Name: bankmarketing_training_pipeline
Version: 1
Created: False
Type: pipeline


In [57]:
import inspect
import pipeline.batch_deployment as batch_deployment

print("========== GET OR CREATE ENDPOINT ==========")
print(inspect.getsource(
    batch_deployment.get_or_create_batch_endpoint
))

print("\n========== GET OR CREATE DEPLOYMENT ==========")
print(inspect.getsource(
    batch_deployment.get_or_create_pipeline_deployment
))

print("\n========== ENSURE DEFAULT DEPLOYMENT ==========")
print(inspect.getsource(
    batch_deployment.ensure_default_deployment
))

========== GET OR CREATE ENDPOINT ==========
def get_or_create_batch_endpoint(
    ml_client,
    endpoint_name,
    allow_creation=False,
):
    """
    Retrieve an existing Batch Endpoint.
    Create one only when explicitly permitted.
    """

    try:
        endpoint = ml_client.batch_endpoints.get(
            name=endpoint_name
        )

        print(f"Reusing existing Batch Endpoint: {endpoint.name}")
        return endpoint

    except ResourceNotFoundError:

        if not allow_creation:
            raise RuntimeError(
                f"Batch Endpoint '{endpoint_name}' does not exist. "
                "Endpoint creation is disabled."
            )

        endpoint = BatchEndpoint(
            name=endpoint_name,
            description="Bank Marketing pipeline endpoint",
        )

        endpoint = ml_client.batch_endpoints.begin_create_or_update(
            endpoint
        ).result()

        print(f"Created Batch Endpoint: {endpoint.name}")
        return endpoint


## 47–49. Retrieve or create Batch Endpoint and deployment

**Objective:** Deploy the reusable pipeline component and set the intended default.

In [58]:
ALLOW_BATCH_ENDPOINT_CREATION = True
ALLOW_BATCH_DEPLOYMENT_CREATION = True
ALLOW_DEFAULT_DEPLOYMENT_UPDATE = True

In [59]:
# ============================================================
# CREATE BATCH ENDPOINT AND DEPLOYMENT
# ============================================================
import pipeline.batch_deployment as batch_deployment

importlib.reload(batch_deployment)

# Create or retrieve Batch Endpoint.
batch_endpoint = batch_deployment.get_or_create_batch_endpoint(
    ml_client,
    settings.BATCH_ENDPOINT_NAME,
    ALLOW_BATCH_ENDPOINT_CREATION,
)

# Create or retrieve pipeline deployment.
batch_pipeline_deployment = (
    batch_deployment.get_or_create_pipeline_deployment(
        ml_client,
        batch_endpoint.name,
        settings.BATCH_DEPLOYMENT_NAME,
        registered_pipeline_component,
        ALLOW_BATCH_DEPLOYMENT_CREATION,
    )
)

# Set the default deployment.
batch_endpoint = batch_deployment.ensure_default_deployment(
    ml_client,
    batch_endpoint,
    batch_pipeline_deployment.name,
    ALLOW_DEFAULT_DEPLOYMENT_UPDATE,
)

print("\n========== BATCH ENDPOINT ==========")
print("Endpoint:", batch_endpoint.name)
print("Deployment:", batch_pipeline_deployment.name)
print("Default deployment:", batch_endpoint.defaults.deployment_name)

Reusing existing Batch Endpoint: bankmarketing-pipeline-endpoint
Reusing existing pipeline deployment.
Deployment: bankmarketing-pipeline-v1
Verified component ID: /subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourceGroups/rg-udacity-azureml/providers/Microsoft.MachineLearningServices/workspaces/udacity-project/components/bankmarketing_training_pipeline/versions/1
Deployment 'bankmarketing-pipeline-v1' is already the default deployment.

========== BATCH ENDPOINT ==========
Endpoint: bankmarketing-pipeline-endpoint
Deployment: bankmarketing-pipeline-v1
Default deployment: bankmarketing-pipeline-v1


## 50–52. Reuse or invoke Batch Endpoint

**Objective:** Use existing successful endpoint-created job or invoke once when enabled.

In [60]:
# ============================================================
# RETRIEVE REGISTERED TRAINING DATA ASSET
# ============================================================

training_data_asset = ml_client.data.get(
    name=settings.DATA_NAME,
    version=settings.DATA_VERSION,
)

print("========== TRAINING DATA ASSET ==========")
print("Name:", training_data_asset.name)
print("Version:", training_data_asset.version)
print("Type:", training_data_asset.type)
print("Path:", training_data_asset.path)

========== TRAINING DATA ASSET ==========
Name: bankmarketing-mltable
Version: 1
Type: mltable
Path: azureml://subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourcegroups/rg-udacity-azureml/workspaces/udacity-project/datastores/workspaceblobstore/paths/LocalUpload/5276f7d1f6f840de14f8e6c1044949b2e2e38c9d2fbd5527815f725a3998172a/bankmarketing_mltable/


In [61]:
# initialize switch
ALLOW_BATCH_INVOCATION = True
ALLOW_BATCH_ENDPOINT_CREATION = False
ALLOW_BATCH_DEPLOYMENT_CREATION = False
ALLOW_DEFAULT_DEPLOYMENT_UPDATE = False

In [62]:
# ALLOW_BATCH_DEPLOYMENT_UPDATE = True

In [63]:
# # ============================================================
# # UPDATE EXISTING PIPELINE BATCH DEPLOYMENT
# # ============================================================

# if not ALLOW_BATCH_DEPLOYMENT_UPDATE:
#     raise RuntimeError("Deployment update is disabled.")

# deployment = PipelineComponentBatchDeployment(
#     name=settings.BATCH_DEPLOYMENT_NAME,
#     endpoint_name=settings.BATCH_ENDPOINT_NAME,
#     component=component,
#     description="Bank Marketing reusable pipeline deployment",
#     settings={
#         "default_compute": settings.COMPUTE_NAME,
#         "continue_on_step_failure": False,
#     },
#     tags={
#         "project": "udacity-project-2",
#         "sdk": "v2",
#     },
# )

# updated_deployment = (
#     ml_client.batch_deployments
#     .begin_create_or_update(deployment)
#     .result()
# )

# print("========== DEPLOYMENT UPDATED ==========")
# print("Name:", updated_deployment.name)
# print("Endpoint:", updated_deployment.endpoint_name)
# print("State:", updated_deployment.provisioning_state)

# # Disable further updates
# ALLOW_BATCH_DEPLOYMENT_UPDATE = False

In [64]:
deployment_check = ml_client.batch_deployments.get(
    name=settings.BATCH_DEPLOYMENT_NAME,
    endpoint_name=settings.BATCH_ENDPOINT_NAME,
)

print("========== UPDATED DEPLOYMENT ==========")
print("Name:", deployment_check.name)
print(
    "Default compute:",
    deployment_check.properties.get("default_compute")
)
print(
    "Component ID:",
    deployment_check.tags.get("PipelineDeployment.ComponentId")
)

========== UPDATED DEPLOYMENT ==========
Name: bankmarketing-pipeline-v1
Default compute: /subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourceGroups/rg-udacity-azureml/providers/Microsoft.MachineLearningServices/workspaces/udacity-project/computes/cpu-cluster
Component ID: /subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourceGroups/rg-udacity-azureml/providers/Microsoft.MachineLearningServices/workspaces/udacity-project/components/bankmarketing_training_pipeline/versions/1


In [65]:
# ============================================================
# RETRY BATCH ENDPOINT INVOCATION
# ============================================================

from azure.ai.ml import Input

# Confirm the registered data asset
training_data_asset = ml_client.data.get(
    name=settings.DATA_NAME,
    version=settings.DATA_VERSION,
)

# Prepare typed inputs
invocation_inputs = {
    "training_data_uri": Input(
        type="uri_folder",
        path=f"azureml:{training_data_asset.name}:{training_data_asset.version}",
    ),
    "target_column": Input(
        type="string",
        default=settings.TARGET_COLUMN,
    ),
}

# Enable invocation for this submission only
ALLOW_BATCH_INVOCATION = True

try:
    invocation_job = ml_client.batch_endpoints.invoke(
        endpoint_name=settings.BATCH_ENDPOINT_NAME,
        deployment_name=settings.BATCH_DEPLOYMENT_NAME,
        inputs=invocation_inputs,
        experiment_name=settings.PIPELINE_EXPERIMENT_NAME,
    )

    print("========== BATCH INVOCATION SUBMITTED ==========")
    print("Job name:", invocation_job.name)
    print("Status:", invocation_job.status)

finally:
    ALLOW_BATCH_INVOCATION = False

========== BATCH INVOCATION SUBMITTED ==========
Job name: pipelinejob-028e7e9f-f153-44b2-89d9-3e4e4f1bf298
Status: Preparing


In [66]:
print("========== BATCH INVOCATION JOB ==========")

job = ml_client.jobs.get(invocation_job.name)

print("Name:", job.name)
print("Status:", job.status)
print("Studio URL:", job.studio_url)

========== BATCH INVOCATION JOB ==========
Name: pipelinejob-028e7e9f-f153-44b2-89d9-3e4e4f1bf298
Status: JobStatus.PREPARING
Studio URL: https://ml.azure.com/runs/pipelinejob-028e7e9f-f153-44b2-89d9-3e4e4f1bf298?wsid=/subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourcegroups/rg-udacity-azureml/workspaces/udacity-project&tid=331ced8a-3e28-4319-a5b6-0da84ff9dfc5


In [67]:
ml_client.jobs.stream(invocation_job.name)

RunId: pipelinejob-028e7e9f-f153-44b2-89d9-3e4e4f1bf298
Web View: https://ml.azure.com/runs/pipelinejob-028e7e9f-f153-44b2-89d9-3e4e4f1bf298?wsid=/subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourcegroups/rg-udacity-azureml/workspaces/udacity-project

Streaming logs/azureml/executionlogs.txt

[2026-11-20 21:08:56Z] Completing processing run id 71122bdc-118d-4bbb-90fc-d7fd13db14ac.
[2026-11-20 21:08:58Z] Completing processing run id ee9256f2-78e7-4a16-8262-f7789fca0885.
[2026-11-20 21:08:59Z] Finishing experiment: no runs left and nothing to schedule.

Execution Summary
RunId: pipelinejob-028e7e9f-f153-44b2-89d9-3e4e4f1bf298
Web View: https://ml.azure.com/runs/pipelinejob-028e7e9f-f153-44b2-89d9-3e4e4f1bf298?wsid=/subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourcegroups/rg-udacity-azureml/workspaces/udacity-project



In [68]:
# A Batch Endpoint invocation can trigger another training pipeline execution
ALLOW_BATCH_INVOCATION = False
ALLOW_BATCH_ENDPOINT_CREATION = False
ALLOW_BATCH_DEPLOYMENT_CREATION = False
ALLOW_BATCH_DEPLOYMENT_UPDATE = False
ALLOW_DEFAULT_DEPLOYMENT_UPDATE = False

In [69]:
# ============================================================
# FINAL BATCH PIPELINE VERIFICATION
# ============================================================

parent_job = ml_client.jobs.get(invocation_job.name)

print("=" * 60)
print("BATCH PIPELINE VERIFICATION")
print("=" * 60)

print("Parent job:", parent_job.name)
print("Parent status:", parent_job.status)

children = list(
    ml_client.jobs.list(parent_job_name=invocation_job.name)
)

print("\n========== CHILD JOBS ==========")

for child in children:
    print("Name:", child.name)
    print("Display name:", child.display_name)
    print("Status:", child.status)
    print("-" * 40)

expected_steps = {"validate_step", "train_step"}

completed_steps = {
    child.display_name
    for child in children
    if str(child.status).lower().endswith("completed")
}

print("\n========== FINAL RESULT ==========")

if (
    str(parent_job.status).lower().endswith("completed")
    and expected_steps.issubset(completed_steps)
):
    print("SUCCESS: Parent pipeline and both child steps completed.")
else:
    print("CHECK REQUIRED: One or more expected steps are missing or incomplete.")

BATCH PIPELINE VERIFICATION
Parent job: pipelinejob-028e7e9f-f153-44b2-89d9-3e4e4f1bf298
Parent status: JobStatus.COMPLETED

========== CHILD JOBS ==========
Name: 71122bdc-118d-4bbb-90fc-d7fd13db14ac
Display name: validate_step
Status: Completed
----------------------------------------
Name: ee9256f2-78e7-4a16-8262-f7789fca0885
Display name: train_step
Status: Completed
----------------------------------------

========== FINAL RESULT ==========
SUCCESS: Parent pipeline and both child steps completed.


In [70]:
# ============================================================
# FINAL BATCH ENDPOINT CONFIGURATION
# ============================================================

endpoint = ml_client.batch_endpoints.get(
    settings.BATCH_ENDPOINT_NAME
)

deployment = ml_client.batch_deployments.get(
    name=settings.BATCH_DEPLOYMENT_NAME,
    endpoint_name=settings.BATCH_ENDPOINT_NAME,
)

print("Endpoint:", endpoint.name)
print("Default deployment:", endpoint.defaults.deployment_name)
print("Deployment:", deployment.name)
print(
    "Default compute:",
    deployment.properties.get("default_compute")
)
print(
    "Component ID:",
    deployment.tags.get("PipelineDeployment.ComponentId")
)

Endpoint: bankmarketing-pipeline-endpoint
Default deployment: bankmarketing-pipeline-v1
Deployment: bankmarketing-pipeline-v1
Default compute: /subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourceGroups/rg-udacity-azureml/providers/Microsoft.MachineLearningServices/workspaces/udacity-project/computes/cpu-cluster
Component ID: /subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourceGroups/rg-udacity-azureml/providers/Microsoft.MachineLearningServices/workspaces/udacity-project/components/bankmarketing_training_pipeline/versions/1


## 53–57. Temporary JobSchedule evidence and immediate disable

**Objective:** Create or enable one schedule only when evidence is missing, then disable after one Completed run.

In [71]:
# ============================================================
# LOAD SCHEDULE MANAGEMENT MODULE
# ============================================================

ALLOW_SCHEDULE_CREATION = True
ALLOW_SCHEDULE_ENABLE = False

import importlib
import inspect
import pipeline.schedule_management as schedule_management

importlib.reload(schedule_management)

print("Schedule module loaded:", schedule_management.__file__)

Schedule module loaded: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/pipeline/schedule_management.py


In [72]:
# ============================================================
# SCHEDULE MANAGEMENT
# ============================================================
import pipeline.schedule_management as schedule_management

importlib.reload(schedule_management)

schedule = schedule_management.get_schedule_or_none(
    ml_client,
    settings.SCHEDULE_NAME,
)

scheduled_job = schedule_management.find_completed_scheduled_job(
    ml_client,
    settings.SCHEDULE_NAME,
)

print("Schedule:", settings.SCHEDULE_NAME)
print("Schedule exists:", schedule is not None)

if schedule is not None:
    print("Schedule enabled:", schedule.is_enabled)

print(
    "Completed scheduled job:",
    scheduled_job.name if scheduled_job else "None",
)

if scheduled_job is not None:

    print("\nCompleted scheduled execution already exists.")
    print("Job:", scheduled_job.name)
    print("Studio URL:", scheduled_job.studio_url)

    if schedule is not None and schedule.is_enabled:
        final_schedule = schedule_management.disable_schedule(
            ml_client,
            settings.SCHEDULE_NAME,
        )
        print("Schedule disabled:", not final_schedule.is_enabled)

else:

    if schedule is None:

        if not ALLOW_SCHEDULE_CREATION:
            raise RuntimeError(
                "Schedule does not exist. "
                "Set ALLOW_SCHEDULE_CREATION=True to create it."
            )

        schedule = schedule_management.create_temporary_schedule(
            ml_client,
            settings.SCHEDULE_NAME,
            pipeline_job_definition,
        )

        print("Temporary schedule created.")

    elif not schedule.is_enabled:

        if not ALLOW_SCHEDULE_ENABLE:
            raise RuntimeError(
                "Schedule exists but is disabled. "
                "Set ALLOW_SCHEDULE_ENABLE=True to enable it."
            )

        schedule = schedule_management.enable_schedule(
            ml_client,
            settings.SCHEDULE_NAME,
        )

        print("Schedule enabled.")

    print(
        "\nSchedule is active. Wait for one scheduled execution "
        "to complete, then rerun this cell."
    )

Schedule: bankmarketing-pipeline-evidence-schedule
Schedule exists: True
Schedule enabled: False
Completed scheduled job: bankmarketing-pipeline-evidence-schedule-08584064011961835976846626091CU90

Completed scheduled execution already exists.
Job: bankmarketing-pipeline-evidence-schedule-08584064011961835976846626091CU90
Studio URL: https://ml.azure.com/runs/bankmarketing-pipeline-evidence-schedule-08584064011961835976846626091CU90?wsid=/subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourcegroups/rg-udacity-azureml/workspaces/udacity-project&tid=331ced8a-3e28-4319-a5b6-0da84ff9dfc5


In [73]:
schedule = schedule_management.get_schedule_or_none(
    ml_client,
    settings.SCHEDULE_NAME,
)

scheduled_job = schedule_management.find_completed_scheduled_job(
    ml_client,
    settings.SCHEDULE_NAME,
)

print("Schedule enabled:", schedule.is_enabled)
print("Scheduled job:", scheduled_job.name if scheduled_job else "Not completed yet")
print("Job status:", scheduled_job.status if scheduled_job else "N/A")

Schedule enabled: False
Scheduled job: bankmarketing-pipeline-evidence-schedule-08584064011961835976846626091CU90
Job status: JobStatus.COMPLETED


In [74]:
# ============================================================
# INSPECT ALL JOBS ASSOCIATED WITH THE SCHEDULE
# ============================================================

print("========== SCHEDULE JOB INSPECTION ==========")

# Retrieve jobs using supported SDK arguments.
all_jobs = list(ml_client.jobs.list())

schedule_jobs = []

for item in all_jobs:

    # Retrieve full job metadata.
    job = ml_client.jobs.get(item.name)

    # Filter by target experiment.
    if (
        getattr(job, "experiment_name", None)
        != settings.PIPELINE_EXPERIMENT_NAME
    ):
        continue

    tags = getattr(job, "tags", {}) or {}
    properties = getattr(job, "properties", {}) or {}

    # Schedule name may be recorded in properties or tags.
    schedule_reference = (
        str(properties.get("azureml.genericTriggerName", ""))
        + " "
        + str(tags)
    )

    if settings.SCHEDULE_NAME.lower() in schedule_reference.lower():

        schedule_jobs.append(job)

        print("\nJob:", job.name)
        print("Status:", job.status)
        print("Experiment:", job.experiment_name)
        print("Schedule:", settings.SCHEDULE_NAME)
        print("Studio URL:", job.studio_url)

        print("Trigger:", properties.get(
            "azureml.genericTriggerName"
        ))

print("\n========== SUMMARY ==========")
print("Schedule:", settings.SCHEDULE_NAME)
print("Matching jobs:", len(schedule_jobs))

if schedule_jobs:
    print("PASS: Schedule-associated jobs identified.")
else:
    print(
        "No matching jobs found. "
        "Inspect job properties and tags before proceeding."
    )

========== SCHEDULE JOB INSPECTION ==========

========== SUMMARY ==========
Schedule: bankmarketing-pipeline-evidence-schedule
Matching jobs: 0
No matching jobs found. Inspect job properties and tags before proceeding.


In [75]:
schedule = ml_client.schedules.get(
    name=settings.SCHEDULE_NAME
)

print("========== CURRENT SCHEDULE ==========")
print("Name:", schedule.name)
print("Enabled:", schedule.is_enabled)
print("Trigger:", schedule.trigger)

print("\n========== TRIGGER DETAILS ==========")

trigger = schedule.trigger

for key, value in vars(trigger).items():
    print(f"{key}: {value}")

========== CURRENT SCHEDULE ==========
Name: bankmarketing-pipeline-evidence-schedule
Enabled: False
Trigger: <azure.ai.ml.entities._schedule.trigger.RecurrenceTrigger object at 0x7a7ad8501870>

========== TRIGGER DETAILS ==========
type: Recurrence
start_time: 2026-11-20 20:14:49
end_time: None
time_zone: UTC
schedule: <azure.ai.ml.entities._schedule.trigger.RecurrencePattern object at 0x7a7ad8503790>
frequency: day
interval: 1


In [76]:
# ============================================================
# CHECK SCHEDULED EXECUTION
# ============================================================

scheduled_job = schedule_management.find_completed_scheduled_job(
    ml_client,
    settings.SCHEDULE_NAME,
)

if scheduled_job is not None:
    print("Job:", scheduled_job.name)
    print("Status:", scheduled_job.status)
    print("Studio URL:", scheduled_job.studio_url)
else:
    print("No completed scheduled execution found yet.")
    print("The scheduled pipeline may still be running.")

Job: bankmarketing-pipeline-evidence-schedule-08584064011961835976846626091CU90
Status: JobStatus.COMPLETED
Studio URL: https://ml.azure.com/runs/bankmarketing-pipeline-evidence-schedule-08584064011961835976846626091CU90?wsid=/subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourcegroups/rg-udacity-azureml/workspaces/udacity-project&tid=331ced8a-3e28-4319-a5b6-0da84ff9dfc5


In [77]:
# ============================================================
# INSPECT RECENT PIPELINE JOBS
# ============================================================

from datetime import datetime, timezone

print("========== RECENT PIPELINE JOBS ==========")

jobs = list(ml_client.jobs.list())

jobs.sort(
    key=lambda j: str(
        getattr(
            getattr(j, "creation_context", None),
            "created_at",
            ""
        )
    ),
    reverse=True,
)

for item in jobs[:20]:

    job = ml_client.jobs.get(item.name)

    print("\n--------------------------------")
    print("Name:", job.name)
    print("Status:", job.status)

    if job.creation_context:
        print(
            "Created:",
            job.creation_context.created_at
        )

    print("Experiment:", getattr(job, "experiment_name", None))
    print("Tags:", getattr(job, "tags", {}))
    print("Properties:", getattr(job, "properties", {}))

========== RECENT PIPELINE JOBS ==========

--------------------------------
Name: pipelinejob-028e7e9f-f153-44b2-89d9-3e4e4f1bf298
Status: JobStatus.COMPLETED
Created: 2026-11-20 21:08:54.350833+00:00
Experiment: bankmarketing-pipeline-sdk-v2
Tags: {'azureml.deploymentname': 'bankmarketing-pipeline-v1', 'azureml.batchrun': 'true', 'azureml.jobtype': 'azureml.pipelinejob'}
Properties: {'azureml.runLineageType': 'batchDeployment', 'azureml.deploymentname': 'bankmarketing-pipeline-v1', 'azureml.endpointname': 'bankmarketing-pipeline-endpoint', 'azureml.SourceComponentId': '/subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourceGroups/rg-udacity-azureml/providers/Microsoft.MachineLearningServices/workspaces/udacity-project/components/bankmarketing_training_pipeline/versions/1', 'azureml.DevPlatv2': 'true', 'azureml.DatasetAccessMode': 'Asset', 'azureml.runsource': 'azureml.PipelineRun', 'runSource': 'BatchDeployment', 'runType': 'HTTP', 'azureml.parameters': '{"target_column":"y"}', 

In [78]:
# ============================================================
# INSPECT SCHEDULE UPDATE METHODS
# ============================================================

schedule = ml_client.schedules.get(
    name=settings.SCHEDULE_NAME
)

print("Schedule:", schedule.name)
print("Enabled:", schedule.is_enabled)
print("Schedule type:", type(schedule))
print("\nAvailable schedule operations:")

print([
    method for method in dir(ml_client.schedules)
    if not method.startswith("_")
])

print("\nSchedule object attributes:")
print([
    attr for attr in dir(schedule)
    if not attr.startswith("_")
])

Schedule: bankmarketing-pipeline-evidence-schedule
Enabled: False
Schedule type: <class 'azure.ai.ml.entities._schedule.schedule.JobSchedule'>

Available schedule operations:
['begin_create_or_update', 'begin_delete', 'begin_disable', 'begin_enable', 'get', 'list', 'service_client', 'trigger']

Schedule object attributes:
['base_path', 'create_job', 'creation_context', 'description', 'display_name', 'dump', 'id', 'is_enabled', 'name', 'properties', 'provisioning_state', 'tags', 'trigger', 'type']


In [79]:
# ============================================================
# DISABLE TEMPORARY EVIDENCE SCHEDULE
# ============================================================

schedule_name = settings.SCHEDULE_NAME

schedule = ml_client.schedules.get(name=schedule_name)

print("Schedule:", schedule.name)
print("Enabled before:", schedule.is_enabled)

# Disable and wait for Azure ML to complete the operation
ml_client.schedules.begin_disable(
    name=schedule_name
).result()

# Verify
schedule = ml_client.schedules.get(name=schedule_name)

print("Enabled after:", schedule.is_enabled)
print("Schedule disabled successfully:", not schedule.is_enabled)

Schedule: bankmarketing-pipeline-evidence-schedule
Enabled before: False
..Enabled after: False
Schedule disabled successfully: True


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


## 58. Document SDK v1-to-v2 mapping

**Objective:** Print the evaluator-facing equivalence statement.

In [80]:
# ============================================================
# 58. DOCUMENT SDK V1-TO-V2 MAPPING
# ============================================================

sdk_mapping = {
    "Workspace": "MLClient",
    "Dataset.Tabular": "MLTable data asset",
    "Experiment and Run": "SDK v2 jobs",
    "AutoMLRun and child runs": "AutoML parent and child jobs",
    "RunDetails": "Studio graph, job status, child jobs, jobs.stream(), and Studio URLs",
    "Model.register": "SDK v2 model asset registration, including MLflow model assets",
    "ACI web service": "Managed Online Endpoint",
    "PublishedPipeline": "Versioned reusable pipeline component",
    "PipelineEndpoint": "Batch Endpoint with pipeline component deployment",
    "Pipeline parameters": "Pipeline component inputs and Batch Endpoint invocation inputs",
    "Published pipeline run": "Pipeline job created by Batch Endpoint invocation",
    "PipelineSchedule": "JobSchedule",
}

print("========== SDK V1 TO V2 MAPPING ==========")

for old, new in sdk_mapping.items():
    print(f"{old} -> {new}")

========== SDK V1 TO V2 MAPPING ==========
Workspace -> MLClient
Dataset.Tabular -> MLTable data asset
Experiment and Run -> SDK v2 jobs
AutoMLRun and child runs -> AutoML parent and child jobs
RunDetails -> Studio graph, job status, child jobs, jobs.stream(), and Studio URLs
Model.register -> SDK v2 model asset registration, including MLflow model assets
ACI web service -> Managed Online Endpoint
PublishedPipeline -> Versioned reusable pipeline component
PipelineEndpoint -> Batch Endpoint with pipeline component deployment
Pipeline parameters -> Pipeline component inputs and Batch Endpoint invocation inputs
Published pipeline run -> Pipeline job created by Batch Endpoint invocation
PipelineSchedule -> JobSchedule


## 59. README and architecture documentation gate

**Objective:** Verify required repository documentation exists before final evidence.

In [81]:
# ============================================================
# 59A. CREATE ARCHITECTURE DOCUMENTATION
# ============================================================

architecture_dir = repo_root / "docs"
architecture_dir.mkdir(parents=True, exist_ok=True)

architecture_path = architecture_dir / "architecture.md"

architecture_content = """# Azure ML Bank Marketing – SDK v2 Architecture

## 1. Project Overview

This project implements an end-to-end machine learning workflow
using Azure Machine Learning SDK v2.

The solution covers data validation, model training, pipeline
orchestration, reusable pipeline components, Batch Endpoint
invocation, and scheduled pipeline execution.

## 2. Architecture Overview

Bank Marketing Dataset
    |
    v
Azure ML MLTable Data Asset
    |
    v
Azure ML SDK v2 Pipeline
    |
    +--> Data Validation
    |        |
    |        v
    |    Validated Data
    |        |
    |        v
    |    Model Training
    |        |
    |        v
    |    Trained Model and Metrics
    |
    +--> Registered Pipeline Component
             |
             v
         Batch Endpoint
             |
             v
         Pipeline Deployment
             |
             v
         Endpoint-Triggered Pipeline Job

JobSchedule
    |
    v
Scheduled Pipeline Job

## 3. Core Azure ML Resources

| Resource | Purpose |
|---|---|
| MLClient | Authenticates and manages Azure ML resources |
| MLTable data asset | Provides versioned training data |
| Pipeline components | Encapsulate reusable processing and training steps |
| Pipeline job | Orchestrates the end-to-end workflow |
| Batch Endpoint | Exposes the reusable pipeline for invocation |
| Batch Deployment | Connects the endpoint to the pipeline component |
| JobSchedule | Triggers recurring pipeline executions |
| CPU compute cluster | Executes pipeline workloads |

## 4. Pipeline Workflow

### Data Validation

- Reads the registered MLTable data asset.
- Validates the input dataset and target column.
- Produces validated data for model training.

### Model Training

- Consumes the validated dataset.
- Trains the classification model.
- Produces model and evaluation metric outputs.

### Pipeline Orchestration

Azure ML SDK v2 pipeline jobs coordinate component execution,
input/output dependencies, compute selection, and job monitoring.

## 5. Operationalization

### Reusable Pipeline Component

The pipeline is registered as a versioned Azure ML component.
This supports reuse without rebuilding the pipeline definition
for every execution.

### Batch Endpoint

A Batch Endpoint exposes the registered pipeline component
through a deployment.

The deployment uses the configured CPU compute cluster.
Invoking the endpoint creates a pipeline job in Azure ML.

### Scheduled Execution

A JobSchedule is configured to trigger the pipeline automatically.

The project includes evidence of a successful scheduled execution.
The temporary evidence schedule was subsequently disabled to
prevent additional recurring executions.

## 6. SDK v1 to SDK v2 Migration

| SDK v1 concept | SDK v2 equivalent |
|---|---|
| Workspace | MLClient |
| Dataset.Tabular | MLTable data asset |
| Experiment and Run | SDK v2 jobs |
| PublishedPipeline | Versioned pipeline component |
| PipelineEndpoint | Batch Endpoint with pipeline deployment |
| PipelineSchedule | JobSchedule |
| RunDetails | Studio job details, graph, logs, and metadata |

## 7. Monitoring and Evidence

Execution evidence is captured through Azure ML Studio and
the project repository.

Evidence includes:

- Completed pipeline graph and component execution.
- Registered pipeline component and version.
- Batch Endpoint and deployment configuration.
- Successful Batch Endpoint invocation.
- Endpoint-created pipeline job.
- Scheduled pipeline execution.
- Disabled final schedule state.

## 8. Repository Organization

The repository separates orchestration code, pipeline components,
source scripts, notebooks, documentation, and evidence.

The notebook coordinates the workflow and documents execution.
Python modules encapsulate reusable pipeline and operationalization
logic.

## 9. Security and Reproducibility

- Credentials and secrets must not be committed.
- Local configuration and generated outputs are excluded from Git.
- Data is referenced through a versioned MLTable asset.
- Azure resource operations are explicitly invoked rather than
  executed as side effects of importing Python modules.
- Screenshots and documentation should be reviewed and redacted
  before publication.
"""

architecture_path.write_text(
    architecture_content.strip() + "\n",
    encoding="utf-8",
)

print("Architecture document created:")
print(architecture_path.relative_to(repo_root))
print("File size:", architecture_path.stat().st_size, "bytes")

Architecture document created:
docs/architecture.md
File size: 4212 bytes


In [82]:
documentation = [
    repo_root / "README.md",
    repo_root / "docs" / "architecture.md",
]

print("========== DOCUMENTATION CHECK ==========")

for p in documentation:
    print(
        "FOUND" if p.exists() else "MISSING",
        p.relative_to(repo_root)
    )

missing_docs = [p for p in documentation if not p.exists()]

if missing_docs:
    raise FileNotFoundError(
        "Required documentation is missing."
    )

print("PASS: Documentation gate completed.")

========== DOCUMENTATION CHECK ==========
FOUND README.md
FOUND docs/architecture.md
PASS: Documentation gate completed.


## 60. Validate Part 2 screenshot inventory

**Objective:** Check required pipeline, endpoint, schedule, and repository evidence.

In [83]:
# ============================================================
# 60. VALIDATE PART 2 SCREENSHOT INVENTORY
# ============================================================

expected_part2 = [
    "01_completed_pipeline_graph.png",
    "02_pipeline_parent_child_jobs.png",
    "03_registered_pipeline_component.png",
    "04_batch_endpoint.png",
    "05_pipeline_component_deployment.png",
    "06_batch_endpoint_invocation.png",
    "07_endpoint_created_pipeline_job.png",
    "08_active_schedule_configuration.png",
    "09_schedule_history.png",
    "10_schedule_created_completed_job.png",
    "11_disabled_schedule.png",
    "12_final_repository_structure.png",
    "13_final_workflow_state.png",
]

print("========== PART 2 SCREENSHOT INVENTORY ==========")

missing_screenshots = []

for name in expected_part2:
    p = repo_root / "screenshots" / "part2" / name

    status = "FOUND" if p.exists() else "MISSING"
    print(status, p.relative_to(repo_root))

    if not p.exists():
        missing_screenshots.append(name)

print("\nScreenshots found:", len(expected_part2) - len(missing_screenshots))
print("Screenshots missing:", len(missing_screenshots))

if missing_screenshots:
    print("\nEvidence still required:")
    for name in missing_screenshots:
        print("-", name)
else:
    print("\nScreenshot inventory complete.")

========== PART 2 SCREENSHOT INVENTORY ==========
MISSING screenshots/part2/01_completed_pipeline_graph.png
MISSING screenshots/part2/02_pipeline_parent_child_jobs.png
MISSING screenshots/part2/03_registered_pipeline_component.png
MISSING screenshots/part2/04_batch_endpoint.png
MISSING screenshots/part2/05_pipeline_component_deployment.png
MISSING screenshots/part2/06_batch_endpoint_invocation.png
MISSING screenshots/part2/07_endpoint_created_pipeline_job.png
MISSING screenshots/part2/08_active_schedule_configuration.png
MISSING screenshots/part2/09_schedule_history.png
MISSING screenshots/part2/10_schedule_created_completed_job.png
MISSING screenshots/part2/11_disabled_schedule.png
MISSING screenshots/part2/12_final_repository_structure.png
MISSING screenshots/part2/13_final_workflow_state.png

Screenshots found: 0
Screenshots missing: 13

Evidence still required:
- 01_completed_pipeline_graph.png
- 02_pipeline_parent_child_jobs.png
- 03_registered_pipeline_component.png
- 04_batch_en

## 61. Screencast checklist

**Objective:** Print the final portfolio-evidence checklist.

In [84]:
# ============================================================
# 61. FINAL PORTFOLIO EVIDENCE CHECKLIST
# ============================================================

screencast_checklist = [
    "1–5 minutes, 1080p or higher, 16:9",
    "Available AutoML model and winning child",
    "Working Managed Online Endpoint prediction",
    "Completed pipeline graph",
    "Registered pipeline component",
    "Batch Endpoint and pipeline deployment",
    "Successful endpoint-created pipeline job",
    "Schedule history and Completed scheduled job",
    "Disabled schedule",
]

print("========== FINAL SCREENCAST CHECKLIST ==========")

for item in screencast_checklist:
    print("[ ]", item)

========== FINAL SCREENCAST CHECKLIST ==========
[ ] 1–5 minutes, 1080p or higher, 16:9
[ ] Available AutoML model and winning child
[ ] Working Managed Online Endpoint prediction
[ ] Completed pipeline graph
[ ] Registered pipeline component
[ ] Batch Endpoint and pipeline deployment
[ ] Successful endpoint-created pipeline job
[ ] Schedule history and Completed scheduled job
[ ] Disabled schedule


## 62. Save final workflow state

**Objective:** Save final selected resource names and disabled-schedule status.

In [85]:
# ============================================================
# RECOVER EXISTING COMPLETED PIPELINE JOBS
# ============================================================

print("========== RECOVER EXISTING JOBS ==========")

# 1. Recover the main completed pipeline job
main_job_name = "pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913"

completed_pipeline_job = ml_client.jobs.get(main_job_name)

print("Main pipeline job:")
print("  Name:", completed_pipeline_job.name)
print("  Status:", completed_pipeline_job.status)


# 2. Recover the completed Batch Endpoint invocation
invocation_name = "pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913"

existing_invocation = ml_client.jobs.get(invocation_name)

print("\nBatch Endpoint invocation:")
print("  Name:", existing_invocation.name)
print("  Status:", existing_invocation.status)


# 3. Recover the completed scheduled execution
schedule_name = settings.SCHEDULE_NAME

scheduled_matches = []

for item in ml_client.jobs.list():
    job = ml_client.jobs.get(item.name)

    properties = getattr(job, "properties", {}) or {}
    trigger_name = properties.get("azureml.genericTriggerName")

    if (
        trigger_name == schedule_name
        and str(job.status).split(".")[-1].lower() == "completed"
    ):
        scheduled_matches.append(job)

if not scheduled_matches:
    raise RuntimeError(
        "No completed scheduled execution found. "
        "Do not submit another job; inspect existing job metadata."
    )

scheduled_pipeline_job = sorted(
    scheduled_matches,
    key=lambda job: str(job.creation_context.created_at)
    if getattr(job, "creation_context", None)
    else "",
    reverse=True,
)[0]

print("\nScheduled pipeline job:")
print("  Name:", scheduled_pipeline_job.name)
print("  Status:", scheduled_pipeline_job.status)

print("\nPASS: Existing job objects recovered.")

========== RECOVER EXISTING JOBS ==========
Main pipeline job:
  Name: pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913
  Status: JobStatus.COMPLETED

Batch Endpoint invocation:
  Name: pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913
  Status: JobStatus.COMPLETED

Scheduled pipeline job:
  Name: bankmarketing-pipeline-evidence-schedule-08584064011961835976846626091CU90
  Status: JobStatus.COMPLETED

PASS: Existing job objects recovered.


In [86]:
scheduled_execution_job = scheduled_pipeline_job

In [87]:
# ============================================================
# 62. SAVE FINAL WORKFLOW STATE
# ============================================================

import json
from datetime import datetime, timezone

print("========== FINAL WORKFLOW VALIDATION ==========")

# Retrieve the final schedule state.
final_schedule = ml_client.schedules.get(
    name=settings.SCHEDULE_NAME
)

# Find the completed job created by this schedule.
scheduled_execution_jobs = []

for item in ml_client.jobs.list():
    job = ml_client.jobs.get(item.name)

    properties = getattr(job, "properties", {}) or {}

    if (
        properties.get("azureml.genericTriggerName")
        == settings.SCHEDULE_NAME
    ):
        scheduled_execution_jobs.append(job)

scheduled_execution_jobs.sort(
    key=lambda job: str(
        getattr(
            getattr(job, "creation_context", None),
            "created_at",
            ""
        )
    ),
    reverse=True,
)

if not scheduled_execution_jobs:
    raise RuntimeError(
        "No job associated with the evidence schedule was found."
    )

scheduled_execution_job = scheduled_execution_jobs[0]


# Validate all three execution paths.
def is_completed(job):
    return str(job.status).split(".")[-1].lower() == "completed"


if not is_completed(completed_pipeline_job):
    raise RuntimeError("Main pipeline job is not Completed.")

if not is_completed(existing_invocation):
    raise RuntimeError("Batch Endpoint invocation is not Completed.")

if not is_completed(scheduled_execution_job):
    raise RuntimeError("Scheduled pipeline execution is not Completed.")

if final_schedule.is_enabled:
    raise RuntimeError(
        "Disable the schedule before saving final workflow state."
    )


# Build final evidence state.
final_state = {
    "saved_utc": datetime.now(timezone.utc).isoformat(),
    "final_status": "COMPLETED",

    "pipeline_job": completed_pipeline_job.name,
    "pipeline_job_status": str(completed_pipeline_job.status),

    "pipeline_component": registered_pipeline_component.name,
    "pipeline_component_version": str(
        registered_pipeline_component.version
    ),

    "batch_endpoint": batch_endpoint.name,
    "batch_deployment": batch_pipeline_deployment.name,

    "endpoint_invocation_job": existing_invocation.name,
    "endpoint_invocation_status": str(existing_invocation.status),

    "scheduled_execution_job": scheduled_execution_job.name,
    "scheduled_execution_status": str(
        scheduled_execution_job.status
    ),

    "schedule": settings.SCHEDULE_NAME,
    "schedule_enabled": final_schedule.is_enabled,
}


# Save JSON.
output_path = repo_root / "outputs" / "final_workflow_state.json"

output_path.parent.mkdir(parents=True, exist_ok=True)

output_path.write_text(
    json.dumps(final_state, indent=2),
    encoding="utf-8",
)

print(json.dumps(final_state, indent=2))
print("\nSaved:", output_path)
print("\nFinal workflow-state validation passed.")

========== FINAL WORKFLOW VALIDATION ==========
{
  "saved_utc": "2026-11-20T21:09:39.228793+00:00",
  "final_status": "COMPLETED",
  "pipeline_job": "pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913",
  "pipeline_job_status": "JobStatus.COMPLETED",
  "pipeline_component": "bankmarketing_training_pipeline",
  "pipeline_component_version": "1",
  "batch_endpoint": "bankmarketing-pipeline-endpoint",
  "batch_deployment": "bankmarketing-pipeline-v1",
  "endpoint_invocation_job": "pipelinejob-f7c9a84e-0adf-4ee0-b555-fb841ae3b913",
  "endpoint_invocation_status": "JobStatus.COMPLETED",
  "scheduled_execution_job": "bankmarketing-pipeline-evidence-schedule-08584064011961835976846626091CU90",
  "scheduled_execution_status": "JobStatus.COMPLETED",
  "schedule": "bankmarketing-pipeline-evidence-schedule",
  "schedule_enabled": false
}

Saved: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/outputs/final_workflow_state.js

## 63. Final cleanup gate

**Objective:** Keep all destructive actions separate and disabled until evidence is secured.

In [88]:
# ============================================================
# 63. FINAL CLEANUP GATE
# ============================================================

if globals().get("ALLOW_CLEANUP", False):
    print(
        "Cleanup enabled. Review each deletion in a separate "
        "visible notebook cell before execution."
    )
else:
    print("Cleanup skipped. ALLOW_CLEANUP is False.")

Cleanup skipped. ALLOW_CLEANUP is False.


# Final Privacy and GitHub Review

Before commit and push:
- Clear notebook outputs containing subscription IDs, tenant IDs, resource IDs, credentials, keys, tokens, private paths, or personal email addresses.
- Confirm `.venv/`, `config.json`, `.env*`, `outputs/`, credentials, keys, and certificates are excluded.
- Keep source code, MLTable definition, safe request sample, README, architecture, and redacted screenshots.
- Verify no module performs Azure resource actions merely on import.
- Commit only after the notebook, source modules, screenshots, README, architecture, and screencast evidence agree.